# EuroSAT External Validation

Uses 400 real and 400 fake images per source class with a fixed stratified split and ResNet-50 preprocessing.


In [6]:
from google.colab import drive
drive.mount('/content/drive')

import os, random, tarfile, json
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn, torch.optim as optim
import torchvision.transforms as transforms, torchvision.models as models
from torchvision.datasets import EuroSAT
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

SEED=42; B=2000; DRIVE='/content/drive/MyDrive'; OUT=f'{DRIVE}/eurosat_clean_rerun'
os.makedirs(OUT,exist_ok=True)

def seed_everything(seed=SEED):
    os.environ['PYTHONHASHSEED']=str(seed); random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
    try: torch.use_deterministic_algorithms(True,warn_only=True)
    except Exception: pass
seed_everything(); device='cuda' if torch.cuda.is_available() else 'cpu'; print('device:',device)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device: cuda


In [7]:
_ = EuroSAT(root='/content/eurosat_real', download=True)
REAL_ROOT='/content/eurosat_real/eurosat/2750'

FAKE_ARCHIVE='/content/eurosat_fake.tar.gz'
if not os.path.exists(FAKE_ARCHIVE):
    import urllib.request
    urllib.request.urlretrieve(
        "https://zenodo.org/record/7861015/files/EuroSat_generated_64.tar.gz?download=1",
        FAKE_ARCHIVE
    )
os.makedirs('/content/eurosat_fake',exist_ok=True)
if not any(os.scandir('/content/eurosat_fake')):
    with tarfile.open(FAKE_ARCHIVE) as t: t.extractall('/content/eurosat_fake')

def find_class_root(base):
    for root,dirs,files in os.walk(base):
        sub=[d for d in dirs if os.path.isdir(os.path.join(root,d))]
        if len(sub)>=5 and any(f.lower().endswith(('.jpg','.png','.jpeg')) for f in os.listdir(os.path.join(root,sub[0]))):
            return root
    return base
FAKE_ROOT=find_class_root('/content/eurosat_fake')
print('real classes:',sorted(os.listdir(REAL_ROOT)))
print('fake classes:',sorted(os.listdir(FAKE_ROOT)))

/tmp/ipykernel_5012/490459576.py:13: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open(FAKE_ARCHIVE) as t: t.extractall('/content/eurosat_fake')


real classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
fake classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'Permanent', 'Residential', 'River', 'SeaLake']


In [8]:
SCENE_MAP={
    'SeaLake':'water','River':'water',
    'AnnualCrop':'agriculture','PermanentCrop':'agriculture','Permanent':'agriculture','Pasture':'agriculture',
    'Forest':'vegetation','HerbaceousVegetation':'vegetation',
    'Industrial':'built','Residential':'built','Highway':'built',
}
# Map Permanent to PermanentCrop.
CANONICAL_CLASS={'Permanent':'PermanentCrop'}
PER_CLASS=400

def collect(root,label):
    rows=[]
    for cls in sorted(os.listdir(root)):
        d=os.path.join(root,cls)
        if not os.path.isdir(d) or cls not in SCENE_MAP: continue
        files=sorted([os.path.join(d,f) for f in os.listdir(d) if f.lower().endswith(('.jpg','.png','.jpeg'))])
        rng=random.Random(SEED + sum(ord(c) for c in cls) + 1000*label)
        rng.shuffle(files)
        canonical=CANONICAL_CLASS.get(cls,cls)
        for p in files[:PER_CLASS]:
            rows.append({'path':p,'label':label,'scene':SCENE_MAP[cls],'source_class':canonical})
    return rows

all_df=pd.DataFrame(collect(REAL_ROOT,0)+collect(FAKE_ROOT,1))
print('total:',len(all_df),'real:',(all_df.label==0).sum(),'fake:',(all_df.label==1).sum())
display(all_df.groupby(['source_class','label']).size().unstack(fill_value=0))
print('\nBroad scenes:')
display(all_df.groupby(['scene','label']).size().unstack(fill_value=0))

# Stratify by source class and real/fake label.
all_df['stratum']=all_df['source_class']+'__'+all_df['label'].astype(str)
train_idx,test_idx=train_test_split(np.arange(len(all_df)),test_size=.2,random_state=SEED,stratify=all_df['stratum'])
all_df['split']='train'; all_df.loc[test_idx,'split']='test'
train_df=all_df[all_df.split.eq('train')].copy(); test_df=all_df[all_df.split.eq('test')].copy()
print('train:',len(train_df),'test:',len(test_df))
display(test_df.groupby(['scene','label']).size().unstack(fill_value=0))
all_df.to_csv(f'{OUT}/eurosat_manifest_seed42.csv',index=False)

total: 8000 real: 4000 fake: 4000


label,0,1
source_class,,
AnnualCrop,400,400
Forest,400,400
HerbaceousVegetation,400,400
Highway,400,400
Industrial,400,400
Pasture,400,400
PermanentCrop,400,400
Residential,400,400
River,400,400



Broad scenes:


label,0,1
scene,,
agriculture,1200,1200
built,1200,1200
vegetation,800,800
water,800,800


train: 6400 test: 1600


label,0,1
scene,,
agriculture,240,240
built,240,240
vegetation,160,160
water,160,160


In [9]:
IMAGENET_MEAN=(0.485,0.456,0.406); IMAGENET_STD=(0.229,0.224,0.225)
transform=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize(IMAGENET_MEAN,IMAGENET_STD)])
class DS(Dataset):
    def __init__(self,frame): self.frame=frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        r=self.frame.iloc[i]
        return transform(Image.open(r.path).convert('RGB')),int(r.label),r.scene,r.path

def loader(frame,shuffle=False):
    g=torch.Generator(); g.manual_seed(SEED)
    return DataLoader(DS(frame),batch_size=32,shuffle=shuffle,generator=g,num_workers=0)

seed_everything()
model=models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
for p in model.parameters(): p.requires_grad=False
model.fc=nn.Linear(model.fc.in_features,2); model=model.to(device)
crit=nn.CrossEntropyLoss(); opt=optim.Adam((p for p in model.parameters() if p.requires_grad),lr=1e-3)
for e in range(5):
    model.train(); run=0
    for x,y,_,_ in loader(train_df,shuffle=True):
        x,y=x.to(device),y.to(device); opt.zero_grad(set_to_none=True); loss=crit(model(x),y); loss.backward(); opt.step(); run+=loss.item()
    print(f'epoch {e+1}/5 loss {run/len(loader(train_df)):.4f}')

torch.save({'state_dict':model.state_dict(),'seed':SEED,'input_size':224,'mean':IMAGENET_MEAN,'std':IMAGENET_STD},f'{OUT}/eurosat_resnet50_seed42.pt')

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 164MB/s]


epoch 1/5 loss 0.0909
epoch 2/5 loss 0.0192
epoch 3/5 loss 0.0113
epoch 4/5 loss 0.0078
epoch 5/5 loss 0.0057


In [10]:
model.eval(); rows=[]
with torch.no_grad():
    for x, y, sc, path in loader(test_df):
        p = torch.softmax(model(x.to(device)), 1)[:,1].cpu().numpy()
        for i in range(len(y)):
            rows.append({
                'path': path[i],
                'label': int(y[i]),
                'scene': sc[i],
                'prob_fake': float(p[i])
            })

pred = pd.DataFrame(rows)
pred['pred'] = (pred.prob_fake >= .5).astype(int)
pred.to_csv(f'{OUT}/eurosat_test_predictions.csv', index=False)

print('overall accuracy:', accuracy_score(pred.label, pred.pred))
print('overall AUC:', roc_auc_score(pred.label, pred.prob_fake))
print('confusion:', confusion_matrix(pred.label, pred.pred).tolist())
print('test real:', int((pred.label == 0).sum()), '| test fake:', int((pred.label == 1).sum()))

def auc_ci(d, seed):
    lab = d.label.to_numpy()
    pro = d.prob_fake.to_numpy()
    idx = np.arange(len(d))
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(B):
        s = rng.choice(idx, len(idx), True)
        if np.unique(lab[s]).size < 2:
            continue
        vals.append(roc_auc_score(lab[s], pro[s]))
    vals = np.asarray(vals)
    return roc_auc_score(lab, pro), np.percentile(vals, 2.5), np.percentile(vals, 97.5)

out = []
for i, sc in enumerate(['water', 'agriculture', 'vegetation', 'built']):
    d = pred[pred.scene.eq(sc)].copy()
    auc, lo, hi = auc_ci(d, SEED + 100 + i)
    out.append({
        'scene': sc,
        'accuracy': float(accuracy_score(d.label, d.pred)),
        'AUC': float(auc),
        'CI_low': float(lo),
        'CI_high': float(hi),
        'n_test': int(len(d)),
        'n_real': int((d.label == 0).sum()),
        'n_fake': int((d.label == 1).sum())
    })

res = pd.DataFrame(out)
display(res)
res.to_csv(f'{OUT}/eurosat_per_scene_results_with_ci_and_counts.csv', index=False)
# Save results.
res.to_csv(f'{OUT}/eurosat_per_scene_results.csv', index=False)

print('\nUse AUC, CI_low and CI_high to add 95% CIs to Table 4.4.')
print('n_real/n_fake are saved so the test composition is explicit and can be reported if needed.')


overall accuracy: 0.998125
overall AUC: 0.999959375
confusion: [[800, 0], [3, 797]]
test real: 800 | test fake: 800


,scene,accuracy,AUC,CI_low,CI_high,n_test,n_real,n_fake
0,water,0.993750,0.99957,0.998461,1.0,320,160,160
1,agriculture,0.997917,1.00000,1.000000,1.0,480,240,240
2,vegetation,1.000000,1.00000,1.000000,1.0,320,160,160
3,built,1.000000,1.00000,1.000000,1.0,480,240,240



Use AUC, CI_low and CI_high to add 95% CIs to Table 4.4.
n_real/n_fake are saved so the test composition is explicit and can be reported if needed.
